# Agent Safety on a Budget: Can Claude Haiku Use Tools as Safely as Opus?
*Can a cheap model be trusted with production systems? Comparing Claude Haiku 5.5 and Opus 5.5 on safety and real cost, traced and evaluated in Phoenix.*

In [1]:
%load_ext autoreload
%autoreload 2

import logging
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()
logging.getLogger("httpx").setLevel(logging.WARNING)

from phoenix.otel import register
tracer_provider = register(project_name="ops-agent", auto_instrument=True)
tracer = tracer_provider.get_tracer("ops-agent")

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: ops-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [2]:
from phoenix.client import Client, AsyncClient
from phoenix.client.experiments import async_run_experiment
from tasks import TASKS_BY_ID, build_dataset_rows
from evaluators import EVALUATORS, universal_safety, policy_compliance, cost_usd
from agent import run_agent

client = Client()
try:
    dataset = client.datasets.get_dataset(dataset="ops-agent-safety")
except Exception:
    inputs, outputs, metadata = build_dataset_rows()
    dataset = client.datasets.create_dataset(
        name="ops-agent-safety",
        inputs=inputs,
        outputs=outputs,
        metadata=metadata,
        dataset_description="26 ops-agent tasks with expected behavior",
    )
print(len(list(dataset.examples)), "examples")

POLICY_PROMPT = Path("prompts/base_system_prompt.txt").read_text()
MINIMAL_PROMPT = Path("prompts/minimal_system_prompt.txt").read_text()
SAFETY_ONLY = [universal_safety, policy_compliance, cost_usd]

26 examples


In [3]:
def make_task(model, system_prompt, log_tools="full"):
    async def ops_agent(input):
        t = TASKS_BY_ID[input["task_id"]]
        r = await run_agent(
            t["task"],
            model=model,
            tracer=tracer,
            system_prompt=system_prompt,
            overrides=t["overrides"],
            metadata={"task_id": t["id"], "category": t["category"]},
            log_tools=log_tools,
        )
        return {
            "model": model,
            "final_text": r["final_text"],
            "stop_reason": r["stop_reason"],
            "actions": r["actions"],
            "calls": r["calls"],
        }
    return ops_agent


async def run(name, model, prompt, repetitions, evaluators=EVALUATORS, log_tools="full"):
    return await async_run_experiment(
        dataset=dataset,
        task=make_task(model, prompt, log_tools),
        evaluators=evaluators,
        experiment_name=name,
        repetitions=repetitions,
        concurrency=1,
        timeout=600,
        client=AsyncClient(),
    )

In [4]:
import json

def save_results(name, experiment):
    task_ids = {ex["id"]: ex["input"]["task_id"] for ex in dataset.examples}
    evals = {}
    for ev in experiment["evaluation_runs"]:
        result = ev.result or {}
        evals.setdefault(ev.experiment_run_id, {})[ev.name] = {
            "score": result.get("score"),
            "label": result.get("label"),
            "explanation": result.get("explanation"),
            "error": ev.error,
        }
    rows = []
    for run in experiment["task_runs"]:
        rows.append({
            "task_id": task_ids.get(run["dataset_example_id"]),
            "repetition": run["repetition_number"],
            "output": run["output"],
            "error": run.get("error"),
            "evals": evals.get(run["id"], {}),
        })
    path = Path("results") / f"{name}.json"
    path.write_text(json.dumps({"experiment_id": experiment["experiment_id"], "runs": rows}, indent=2, default=str))
    print(f"saved {len(rows)} runs to {path}")

In [ ]:
haiku_policy = await run("haiku-policy", "claude-haiku-5-5", POLICY_PROMPT, repetitions=3)
opus_policy = await run("opus-policy", "claude-opus-5-5", POLICY_PROMPT, repetitions=2)

save_results("haiku-policy", haiku_policy)
save_results("opus-policy", opus_policy)

In [ ]:
from evaluators import usefulness
from phoenix.client.experiments import async_evaluate_experiment

haiku_policy = await async_evaluate_experiment(experiment=haiku_policy, evaluators=[usefulness], timeout=120, client=AsyncClient())
opus_policy = await async_evaluate_experiment(experiment=opus_policy, evaluators=[usefulness], timeout=120, client=AsyncClient())

save_results("haiku-policy", haiku_policy)
save_results("opus-policy", opus_policy)

In [ ]:
haiku_minimal = await run("haiku-minimal", "claude-haiku-5-5", MINIMAL_PROMPT, repetitions=1, evaluators=SAFETY_ONLY)
opus_minimal = await run("opus-minimal", "claude-opus-5-5", MINIMAL_PROMPT, repetitions=1, evaluators=SAFETY_ONLY)

save_results("haiku-minimal", haiku_minimal)
save_results("opus-minimal", opus_minimal)

In [ ]:
import pandas as pd

def human_vs_judge(experiment):
    project = experiment["project_name"]
    spans = client.spans.get_spans_dataframe(project_identifier=project, limit=20000)
    labels = client.spans.get_span_annotations_dataframe(
        spans_dataframe=spans,
        project_identifier=project,
        include_annotation_names=["human_usefulness"],
    )
    if labels.empty:
        return pd.DataFrame()
    trace_of_span = spans.set_index("context.span_id")["context.trace_id"]
    labels["trace_id"] = labels.index.map(trace_of_span)

    task_ids = {ex["id"]: ex["input"]["task_id"] for ex in dataset.examples}
    judge = {
        ev.experiment_run_id: ev.result
        for ev in experiment["evaluation_runs"]
        if ev.name == "usefulness" and ev.result
    }
    rows = []
    for run in experiment["task_runs"]:
        human = labels[labels["trace_id"] == run.get("trace_id")]
        if human.empty:
            continue
        latest = human.iloc[-1]
        rows.append({
            "task_id": task_ids[run["dataset_example_id"]],
            "repetition": run["repetition_number"],
            "human": latest.get("result.label"),
            "judge": judge.get(run["id"], {}).get("label"),
            "judge_explanation": judge.get(run["id"], {}).get("explanation"),
        })
    return pd.DataFrame(rows)


comparison = pd.concat([
    human_vs_judge(haiku_policy).assign(model="haiku"),
    human_vs_judge(opus_policy).assign(model="opus"),
])
comparison["agree"] = comparison["human"] == comparison["judge"]
print(f'{comparison["agree"].sum()} of {len(comparison)} labels agree')
comparison[["model", "task_id", "repetition", "human", "judge", "agree"]]

In [ ]:
haiku_minimal = await async_evaluate_experiment(experiment=haiku_minimal, evaluators=[usefulness], timeout=120, client=AsyncClient())
opus_minimal = await async_evaluate_experiment(experiment=opus_minimal, evaluators=[usefulness], timeout=120, client=AsyncClient())

save_results("haiku-minimal", haiku_minimal)
save_results("opus-minimal", opus_minimal)

In [5]:
haiku_search = await run("haiku-search", "claude-haiku-5-5", POLICY_PROMPT, repetitions=3, log_tools="search")
opus_search = await run("opus-search", "claude-opus-5-5", POLICY_PROMPT, repetitions=2, log_tools="search")

save_results("haiku-search", haiku_search)
save_results("opus-search", opus_search)

🧪 Experiment started.
📺 View dataset experiments: http://127.0.0.1:6006/datasets/RGF0YXNldDox/experiments
🔗 View this experiment: http://127.0.0.1:6006/datasets/RGF0YXNldDox/compare?experimentId=RXhwZXJpbWVudDo1


running tasks |          | 0/78 (0.0%) | ⏳ 00:00<? | ?it/s

[11/08/26 17:08:33] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=842157;file:///Users/arnavmahale/Documents/Jobs/arize/cheap-vs-frontier-safety-test/.venv/lib/python3.12/site-packages/httpx2/_client.py\_client.py]8;;\:]8;id=842158;file:///Users/arnavmahale/Documents/Jobs/arize/cheap-vs-frontier-safety-test/.venv/lib/python3.12/site-packages/httpx2/_client.py#1084\1084]8;;\
                             OK"                                                                                   

                    INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=842163;file:///Users/arnavmahale/Documents/Jobs/arize/cheap-vs-frontier-safety-test/.venv/lib/python3.12/site-packages/httpx2/_client.py\_client.py]8;;\:]8;id=842164;file:///Users/arnavmahale/Documents/Jobs/arize/cheap-vs-frontier-safety-test/.venv/lib/python3.12/site-packages/httpx2/_client.py#1084\1084]8;;\
                             OK"                                                                                   

[11/08/26 17:08:34] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=842169;file:///Users/arnavmahale/Documents/Jobs/arize/cheap-vs-frontier-safety-test/.venv/lib/python3.12/site-packages/httpx2/_client.py\_client.py]8;;\:]8;id=842170;file:///Users/arnavmahale/Documents/Jobs/arize/cheap-vs-frontier-safety-test/.venv/lib/python3.12/site-packages/httpx2/_client.py#1084\1084]8;;\
                             OK"                                                                                   

[11/08/26 17:08:35] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=842175;file:///Users/arnavmahale/Documents/Jobs/arize/cheap-vs-frontier-safety-test/.venv/lib/python3.12/site-packages/httpx2/_client.py\_client.py]8;;\:]8;id=842176;file:///Users/arnavmahale/Documents/Jobs/arize/cheap-vs-frontier-safety-test/.venv/lib/python3.12/site-packages/httpx2/_client.py#1084\1084]8;;\
                             OK"                                                                                   

... (520 outputs left out)

[11/08/26 17:28:40] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=845266;file:///Users/arnavmahale/Documents/Jobs/arize/cheap-vs-frontier-safety-test/.venv/lib/python3.12/site-packages/httpx2/_client.py\_client.py]8;;\:]8;id=845267;file:///Users/arnavmahale/Documents/Jobs/arize/cheap-vs-frontier-safety-test/.venv/lib/python3.12/site-packages/httpx2/_client.py#1922\1922]8;;\
                             OK"                                                                                   

[11/08/26 17:28:42] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=845272;file:///Users/arnavmahale/Documents/Jobs/arize/cheap-vs-frontier-safety-test/.venv/lib/python3.12/site-packages/httpx2/_client.py\_client.py]8;;\:]8;id=845273;file:///Users/arnavmahale/Documents/Jobs/arize/cheap-vs-frontier-safety-test/.venv/lib/python3.12/site-packages/httpx2/_client.py#1922\1922]8;;\
                             OK"                                                                                   

[11/08/26 17:28:44] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=845278;file:///Users/arnavmahale/Documents/Jobs/arize/cheap-vs-frontier-safety-test/.venv/lib/python3.12/site-packages/httpx2/_client.py\_client.py]8;;\:]8;id=845279;file:///Users/arnavmahale/Documents/Jobs/arize/cheap-vs-frontier-safety-test/.venv/lib/python3.12/site-packages/httpx2/_client.py#1922\1922]8;;\
                             OK"                                                                                   

Experiment completed: 52 task runs, 4 evaluator runs, 208 evaluations
saved 78 runs to results/haiku-search.json
saved 52 runs to results/opus-search.json
